In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path: sys.path.append(module_path)
    
from ytmusic_library import YTMusicPlaylists

RUN_API_AUTH_TEST = True
HEADER_FILE = '../oauth.json'
PLAYLIST_TSV_DIR = '../playlists/'

In [2]:
Y = YTMusicPlaylists(header=HEADER_FILE, playlist_tsv_dir=PLAYLIST_TSV_DIR)
if RUN_API_AUTH_TEST: Y.test_ytmusic_api()
print(f"Loaded {len(Y.playlists['title'].unique())} playlists")

Using header file: ../oauth.json
Test Passed in 3.61 seconds
Using ytmusicapi version: 1.7.0
Loaded 506 playlists


# Clean Up Radio Playlists

* Move LIKE to radios like playlist
* Remove DISLIKE and NOT LIKE

In [ ]:
playlist_names = [
]

MIN_RADIO_LIKE_TO_SPLIT=2
for playlist_name in playlist_names:
    assert 'radio' in playlist_name
    pl_info = Y.playlist_get_info(Y.query_by_title(playlist_name).playlistId)
    clean_counters = Y.clean_up_radio_playlist(pl_info, 
        move_like=True, create_like_playlist=True, remove_dislike=True, remove_not_like=True,
        min_num_like=MIN_RADIO_LIKE_TO_SPLIT,  sleep=1, verbose=True,
    )


# Re-Sort Playlist based on LastFM playcount 

creates new sorted pl, deletes old


In [ ]:
playlist_names = [
]
USE_CACHE=False
for playlist_name in playlist_names:
    pl_info = Y.playlist_get_info(Y.query_by_title(playlist_name).playlistId, use_cache=USE_CACHE)
    pc_df = Y.playcount_sort_playlist(pl_info, ignore_banned=True)


## Save playlist backup tsv

In [5]:
PLAYLIST_NAME = 'zz not like 2'
USE_CACHE = False
_pl_id = Y.query_by_title(PLAYLIST_NAME).playlistId
_pl_tracks, _pl_metadata = Y.save_playlist_tsv(Y.playlist_get_info(_pl_id, use_cache=USE_CACHE))

                                                   0
owned                                           True
id                PLWptjpDqazOymmTOH-SrVAbtGxYiosu73
privacy                                      PRIVATE
description                                     None
title                                  zz not like 2
artists                                           []
year                                            2024
views                                           None
duration                                    7+ hours
trackCount                                      3169
related                                           []
duration_seconds                              759762


## Update _not_like tsv

In [6]:
not_like_tracks = Y.collect_all_not_like_tracks_from_tsvs()
not_like_tracks.to_csv(Y.not_like_tsv, sep='\t', header=True)
print(f'Saved {len(not_like_tracks)} entries to not_like tsv: {Y.not_like_tsv}')

Found 2 not like playlists out of the 573 total
Updated not liked tracks, contains 7970 entries.
Saved 7970 entries to not_like tsv: ../playlists/_not_liked_tracks.tsv


## Update _like tsv and print like coverage

In [7]:
like_tracks = Y.collect_all_like_tracks_from_tsvs()
like_tracks.to_csv(Y.like_tsv, sep='\t', header=True)
print(f'Saved {len(not_like_tracks)} entries to like tsv: {Y.like_tsv}')

Found 281 like playlists out of the 573 total
Beats Lofi.tsv	94.0% currently liked (of 100 total tracks)
Beats Without Rhymes.tsv	88.9% currently liked (of 45 total tracks)
Beats indie Chill.tsv	95.2% currently liked (of 83 total tracks)
Bossa Nova.tsv	95.2% currently liked (of 21 total tracks)
Brass n chill.tsv	95.7% currently liked (of 258 total tracks)
Chillwave.tsv	94.0% currently liked (of 367 total tracks)
Folk.tsv	99.2% currently liked (of 249 total tracks)
Grunge.tsv	100.0% currently liked (of 79 total tracks)
Hip Hop 1980s.tsv	100.0% currently liked (of 25 total tracks)
Hip Hop 1990s G funk.tsv	100.0% currently liked (of 28 total tracks)
Hip Hop 1990s east coast.tsv	100.0% currently liked (of 29 total tracks)
Hip Hop 1990s gangst.tsv	100.0% currently liked (of 48 total tracks)
Hip Hop 1990s west Coast.tsv	100.0% currently liked (of 73 total tracks)
Hip Hop 1990s.tsv	98.2% currently liked (of 620 total tracks)
Hip Hop 2000s southern.tsv	100.0% currently liked (of 25 total track

## Get Public Playlists

In [8]:
Y.get_playlists_by_privacy(privacy='PUBLIC')
# last run 4-2024, 3 public playlists

Found public playlist named: Liked Music
Found public playlist named: indie loose
Found public playlist named: Chill Supermix
Found public playlist named: Episodes for Later


title                                                Liked Music
playlistId                                                    LM
thumbnails     [{'url': 'https://www.gstatic.com/youtube/medi...
description                                        Auto playlist
count                                                        NaN
author                                                       NaN
title                                                indie loose
playlistId                    PLWptjpDqazOygyoaRpT447v7SENKpPNfi
thumbnails     [{'url': 'https://yt3.googleusercontent.com/tZ...
description                                Jake G • 1,584 tracks
count                                                      1,584
author         [{'name': 'Jake G', 'id': 'UCDvJYHQoKPhpF-sGFU...
title                                             Chill Supermix
playlistId           RDTMAK5uy_nzfwl2UYv7htL7wDoxbX8Pp6UAFBd92cQ
thumbnails     [{'url': 'https://music.youtube.com/image/mixa...
description              

## Get Playlist Counts

In [ ]:
%%time
playlist_file = os.path.join(PLAYLIST_TSV_DIR, '_playlist_radio_counts.tsv')
playlists = Y.get_playlist_counts(verbose=False, filter_title='radio')
playlists.to_csv(playlist_file, sep='\t', index=False)
# Note: now part of ytmusic backup

playlists.head(20)
# last run 5-5-2024

# One Off

## Upload playlist from tsv backup

In [8]:
PLAYLIST_NAME = 'oldies 1930-40s naptime radio'
playlist_file = os.path.join(PLAYLIST_TSV_DIR, PLAYLIST_NAME + '.tsv')
Y.playlist_from_tsv(playlist_file, ignore_banned=True, sort_by_index=True)


Generating oldies 1930-40s naptime radio ytmusic playlist for 32 tracks
Saved 32 oldies 1930-40s naptime radio tracks playlist with id: PLWptjpDqazOyCYHZ9kLX-ME6LY12iwtlA


## Query

In [ ]:
Y.query_by_title(playlist_name)

title                                                 jazz radio
playlistId                    PLWptjpDqazOxgrXKM4xfJ-SBRahEDfXmG
thumbnails     [{'url': 'https://yt3.googleusercontent.com/zS...
description                                  Jake G • 797 tracks
count                                                        797
author         [{'name': 'Jake G', 'id': 'UCDvJYHQoKPhpF-sGFU...
Name: 20, dtype: object

### Generate playlist froma a list of albums

In [6]:
name = 'y_2023_albums_to_listen_to_v3'
desc = 'manually selected albums to top off 2023 albums'
albums_to_add = [
"Aesop Rock - Integrated Tech Solutions",
"Mitski - The Land Is Inhospitable and So Are We",
"Olivia Rodrigo - GUTS",
"The National - First Two Pages Of Frankenstein",
"Foo Fighters - But Here We Are",
"Jessie Ware - That! Feels Good!",
"Paramore - This Is Why",
"Skrillex - Quest for Fire",
"Fever Ray - Radical Romantics",
"Feist - Multitudes",
"Julie Byrne - The Greater Wings",
"PJ Harvey - I Inside The Old Year Dying",
"Depeche Mode - Memento Mori",
"DJ Shadow - Action Adventure",
"Hania Rani - Ghosts",
"Mary Lattimore - Goodbye, Hotel Arkada",
"Nas - Magic 2",
"Temples - Exotico",
"Nas - Magic 3",
"Everything but the Girl - Fuse",
"Yves Tumor - Praise A Lord Who Chews But Which Does Not Consume (Or Simply, Hot Between Worlds)",
"Sampha - Lahai",
"Noname - Sundial",
"Shame - Food For Worms",
"The Mountain Goats - Jenny from Thebes",
"James Blake - Playing Robots Into Heaven",
"Ratboys - The Window",
"McKinley Dixon - Beloved! Paradise! Jazz!?",
"Taylor Swift - Speak Now (Taylor's Version)",
"Travis Scott - Utopia",
"Water from Your Eyes - Everyone's Crushed",
"Jason Isbell and The 400 Unit - Weathervanes",
"Slowthai - Ugly",
"James Holden - Imagine This Is a High Dimensional Space of All Possibilities",
"The National - Laugh Track",
"Armand Hammer - We Buy Diabetic Test Strips",
"Jeff Rosenstock - Hellmode",
"Hania Rani - On Giacometti",
"Animal Collective - Isn't It Now?",
"Model/Actriz - Dogsbody",
"Sleaford Mods - UK GRIM",
"Belle And Sebastian - Late Developers",
"Hannah Diamond - Perfect Picture",
"The Rolling Stones - Hackney Diamonds",
"Black Thought & El Michels Affair - Glorious Game",
"Noel Gallagher's High Flying Birds - Council Skies",
"Susanne Sundfør - Blómi",
"Arooj Aftab - Love In Exile",
"Lonnie Holley - Oh Me Oh My",
"Tomb Mold - The Enduring Spirit",
"George Clanton - Ooh Rap I Ya",
"Lisa O'Neill - All Of This Is Chance",
"Liturgy - 93696",
"Peter Gabriel - I/O",
"Yussef Dayes - Black Classical Music",
"Loraine James - Gentle Confrontation",
"amaarae - Fountain Baby",
"Kali Uchis - Red Moon in Venus",
"M83 - Fantasy",
"Boygenius - The Record",
"Beirut - Hadsel",
"Yeule - Softscars",
"Osees - Intercepted Message",
"Zach Bryan - Zach Bryan",
"Squid - O Monolith",
"Weval - Remember",
"The Gaslamp Killer & The Heliocentrics - Legna",
"The Clientele - I Am Not There Anymore",
"Beach Fossils - Bunny",
"Laurel Halo - Atlas",
"Youth Lagoon - Heaven Is a Junkyard",
"a.s.o. - a.s.o.",
"Aphex Twin - Blackbox Life Recorder 21f / in a room7 F760",
"Orbital - Optical Delusion",
"Blackbraid - Blackbraid II",
"Ryuichi Sakamoto (坂本龍一) - 12",
"Lankum - False Lankum",
"Metric - Formentera II",
"Barry Can't Swim - When Will We Land?",
"Steven Wilson - The Harmony Codex",
"Black Pumas - Chronicles of a Diamond",
"Laurent Garnier - 33 tours et puis s'en vont",
"Daughter - Stereo Mind Game",
"Earl Sweatshirt & The Alchemist - Voir Dire",
"Czarface - Czartificial Intelligence",
"In Flames - Foregone",
"Eluvium - (Whirring Marvels In) Consensus Reality",
"The Orb - Prism",
"GUNSHIP - Unicorn",
"feeble little horse - Girl with Fish",
"Jaimie Branch - Fly or Die Fly or Die Fly or Die ((world war))",
"Blondshell - Blondshell",
"Nourished by Time - Erotic Probiotic 2",
"Bar Italia - Tracey Denim",
"Ladytron - Time's Arrow",
"Haken - Fauna",
"Carbon Based Lifeforms - Seeker",
"The Ocean - Holocene",
"NewJeans (뉴진스) - Get Up",
"Protomartyr - Formal Growth in the Desert",
"Le Cri du Caire - Le Cri du Caire",
"Kylie Minogue - Tension",
"Alfa Mist - Variables",
"††† - Goodnight, God Bless, I Love U, Delete.",
"Colin Stetson - When We Were That What Wept for the Sea",
"Sleep Token - Take Me Back To Eden",
"Jeromes Dream - The Gray In Between",
"Blockhead - The Aux",
"Jane Remover - Census Designated",
"Portugal. The Man - Chris Black Changed My Life",
"Pangaea - Changing Channels",
"Anthony Naples - orbs",
"Biosphere - Inland Delta",
"Khruangbin - Live at Sydney Opera House",
"Actress - LXXXVIII",
"blink-182 - One More Time...",
"Katatonia - Sky Void of Stars",
"Overmono - Good Lies",
"underscores - Wallsocket"
]

# track_ids = []
for a in albums_to_add:
    match = {}
    res = Y.yt.search(query=a, filter='albums', limit=1)
    result_album = f"{res[0]['artists'][0]['name']} - {res[0]['title']}"
    if len(res) == 0 or res[0].get('browseId') == None:
        print(f'Skipping query: {a} bad result: {res}')
        continue
    yt_album = Y.yt.get_album(res[0]['browseId'])
    for t in yt_album['tracks']:
        track_ids.append(t['videoId'])
    print(f'Added {len(yt_album["tracks"])} tracks":\n\tq: {a}\n\tr: {result_album}')

pl_id = Y.yt.create_playlist( title=name,  description=desc, video_ids=track_ids)
print(f'Generated playlist: {name} with id {pl_id}')

Added 17 tracks":
	q: blink-182 - One More Time...
	r: blink-182 - ONE MORE TIME...
Added 11 tracks":
	q: Katatonia - Sky Void of Stars
	r: Katatonia - Sky Void of Stars
Added 13 tracks":
	q: Overmono - Good Lies
	r: Overmono - Good Lies
Added 12 tracks":
	q: underscores - Wallsocket
	r: underscores - Wallsocket
Generated playlist: y_2023_albums_to_listen_to_v3 with id PLWptjpDqazOwdLiofhSWKIB4yw0DRZqpe
